In [4]:
#Libraries
# Install necessary libraries. NLTK is for basic NLP, Scikit-learn for TF-IDF/similarity.
# NetworkX is essential for the TextRank graph algorithm.
!pip install nltk scikit-learn networkx

# Import core libraries
import nltk
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# Download NLTK data (needed for tokenization and sentence splitting)
nltk.download('punkt')
nltk.download('punkt_tab') # <-- This is the fix!
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [22]:
#Load sample data
# --- SIMULATED DATASET ---
data = {
    'article_id': [1],
    'text': [
        "The recent study on climate change revealed alarming trends. Global average temperatures have risen by 1.2 degrees Celsius since pre-industrial times. This warming is primarily driven by human activity, notably the burning of fossil fuels, which releases greenhouse gases into the atmosphere. The consequences are evident in extreme weather events, including more frequent heatwaves and intense floods. Scientists emphasize that immediate and coordinated international action is required to mitigate these effects. Transitioning to renewable energy sources like solar and wind is a critical step. Delaying action will significantly increase the long-term costs and severity of impacts. Furthermore, biodiversity loss is accelerating, closely tied to these climatic shifts."
    ]
}
df = pd.DataFrame(data)

# Define the article we will summarize
ARTICLE = df['text'][0]

print("--- Original Article ---")
print(ARTICLE)
print("-" * 30)
print(f"Article Length: {len(ARTICLE.split())} words")

--- Original Article ---
The recent study on climate change revealed alarming trends. Global average temperatures have risen by 1.2 degrees Celsius since pre-industrial times. This warming is primarily driven by human activity, notably the burning of fossil fuels, which releases greenhouse gases into the atmosphere. The consequences are evident in extreme weather events, including more frequent heatwaves and intense floods. Scientists emphasize that immediate and coordinated international action is required to mitigate these effects. Transitioning to renewable energy sources like solar and wind is a critical step. Delaying action will significantly increase the long-term costs and severity of impacts. Furthermore, biodiversity loss is accelerating, closely tied to these climatic shifts.
------------------------------
Article Length: 107 words


In [23]:
#Text preprocessing
from nltk.corpus import stopwords
import re

# Initialize stop words and sentence tokenizer
STOP_WORDS = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Convert to Lowercase and remove special characters
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s\.]', '', text) # Keep only letters, numbers, spaces, and periods (for sentence split)

    # 2. Tokenize into sentences
    sentences = nltk.sent_tokenize(text)

    # 3. Clean sentences for vectorization (remove stopwords, extra spaces)
    cleaned_sentences = []
    for sent in sentences:
        # Tokenize words, filter stop words and single-character tokens
        words = [word for word in sent.split() if word not in STOP_WORDS and len(word) > 1]
        cleaned_sentences.append(" ".join(words))

    return sentences, cleaned_sentences

# Run Preprocessing
original_sentences, cleaned_sentences = preprocess_text(ARTICLE)

print(f"Total Sentences: {len(original_sentences)}")
print("\n--- Sample Original Sentence ---")
print(original_sentences[0])
print("\n--- Sample Cleaned Sentence (for vectorization) ---")
print(cleaned_sentences[0])

Total Sentences: 8

--- Sample Original Sentence ---
the recent study on climate change revealed alarming trends.

--- Sample Cleaned Sentence (for vectorization) ---
recent study climate change revealed alarming trends.


In [24]:
#Feature extraction
# Initialize TF-IDF Vectorizer
# Use cleaned_sentences for vectorization since stopwords are removed
vectorizer = TfidfVectorizer()

# Fit and transform the cleaned sentences
# This creates a matrix where each row is a sentence and each column is a word's TF-IDF score
sentence_vectors = vectorizer.fit_transform(cleaned_sentences)

# Get the dense array for easier calculation
sentence_vectors = sentence_vectors.toarray()

print(f"Shape of Sentence Vectors Matrix: {sentence_vectors.shape}")
print("Rows = Sentences, Columns = Unique Words (Features)")

Shape of Sentence Vectors Matrix: (8, 72)
Rows = Sentences, Columns = Unique Words (Features)


In [25]:
#Text rank ,sentense scoring
# ... (Previous code to calculate scores and similarity_matrix)

# 3. Apply PageRank (TextRank) Algorithm
scores = nx.pagerank(text_graph)

# --- ADDING POSITIONAL WEIGHTING ---
# Give the first sentence (index 0) a small bonus, as it often states the topic.
positional_boost = 0.015
scores[0] += positional_boost
# You could also add a smaller boost to index 1, etc., if desired.

# Convert scores into a list of (score, index) pairs
# Recalculate ranked_sentences with the new score
ranked_sentences = sorted([(scores[i], i) for i in range(len(scores))], reverse=True)


print("--- Top 3 Sentence Scores and Indices (Score, Index) ---")
print(ranked_sentences[:3])

--- Top 3 Sentence Scores and Indices (Score, Index) ---
[(0.14, 0), (0.125, 7), (0.125, 5)]


In [26]:
#Summary Generation
# Assuming 'original_sentences' (from Cell 3) and 'ranked_sentences' (from Cell 5)
# are available because the preceding cells have been run.

# Set the desired summary length
N_SENTENCES = 3

# 1. Select the indices of the top N sentences from the ranked list
# This uses the result of the TextRank scoring from Cell 5: [(0.14, 0), (0.125, 7), (0.125, 5)]
top_sentence_indices = [index for score, index in ranked_sentences[:N_SENTENCES]]

# 2. Sort the indices by their original position in the article for coherence.
top_sentence_indices.sort()
# Resulting indices: [0, 5, 7]

# 3. Generate the final summary by joining the original sentences.
# This line is the one causing the error if 'original_sentences' is empty/missing.
final_summary = [original_sentences[i] for i in top_sentence_indices]
final_summary = " ".join(final_summary)

# --- Output ---
print(f"Requested Summary Length: {N_SENTENCES} sentences")
print("\n--- Extractive Summary (TextRank with Positional Weighting) ---")
print(final_summary)

Requested Summary Length: 3 sentences

--- Extractive Summary (TextRank with Positional Weighting) ---
the recent study on climate change revealed alarming trends. transitioning to renewable energy sources like solar and wind is a critical step. furthermore biodiversity loss is accelerating closely tied to these climatic shifts.
